In [ ]:
"""
churn_predictor.py  —  Subspace user churn prediction
Features: payment_failure_count, support_ticket_count,
  days_since_last_active, group_size, subscription_count,
  avg_renewal_delay_days, gift_card_failure_flag
Target: churned_30d (1 = user leaves within 30 days)

From Play Store data, the highest-signal churn indicators are:
  - gift card fulfillment failure (immediate trust collapse)
  - support_ticket_unresolved > 48h
  - group admin removing user (bilateral churn)
"""
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = [
    "payment_failure_count_30d",     # #failures in last 30d
    "support_ticket_unresolved_48h",  # binary: ticket open > 48h
    "days_since_last_active",         # recency
    "active_group_count",             # network depth
    "gift_card_failure_flag",         # binary: any GC failure
    "avg_renewal_delay_days",         # behavioral friction proxy
    "admin_removal_count_90d",        # trust signal from group admins
    "tenure_days",                    # total user age
    "subscription_count_active",      # depth of engagement
    "wallet_zero_balance_streak",     # days wallet at ₹0
]

def build_churn_model(df: pd.DataFrame) -> xgb.XGBClassifier:
    X = df[FEATURES]
    y = df["churned_30d"]

    # Class imbalance: churned users ~15–20% of base
    scale_pos = (y == 0).sum() / (y == 1).sum()

    model = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,   # handle class imbalance
        eval_metric="auc",
        use_label_encoder=False,
        random_state=42
    )
    # 5-fold stratified CV — report mean AUC
    cv_scores = cross_val_score(
        model, X, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True),
        scoring="roc_auc", n_jobs=-1
    )
    print(f"CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    model.fit(X, y, verbose=False)
    return model

def churn_risk_intervention(model: xgb.XGBClassifier, user_row: pd.DataFrame) -> dict:
    """
    Given a user's feature row, return churn probability
    and recommended intervention tier.
    Tiers:
      <0.3  : No action
      0.3–0.6: In-app nudge (wallet cashback ₹20)
      0.6–0.8: Push notification + discount offer
      >0.8  : Human call within 2h
    """
    prob = model.predict_proba(user_row[FEATURES])[:,1][0]
    interventions = [
        (0.80, "HUMAN_CALL",    "Call user within 2h, offer 1-month free"),
        (0.60, "PUSH_DISCOUNT", "Push: 'Exclusive deal for you — 30% off renewal'"),
        (0.30, "WALLET_NUDGE",  "In-app: ₹20 cashback added to your wallet"),
        (0.00, "NO_ACTION",     "Monitor"),
    ]
    tier, action = next(
        (t, a) for threshold, t, a in interventions
        if prob >= threshold
    )
    return {"churn_prob": round(prob, 3), "intervention": tier, "action": action}